# Traffic Congestion Prediction and Accident Risk Analysis
ADA23CT301 Deep Learning - project notebook

**Run the cells from top to bottom.** Runtime > Change runtime type > **GPU** first.

**Before you start, make this folder in your Google Drive:** `MyDrive/traffic_project/`
- `images/` : your 100-300 traffic photos (.jpg / .png)
- `labels.csv` : two columns, `filename,level` (level = low / medium / high). Label by looking at each photo yourself.
- `accidents.csv` : a sample (50k-100k rows) of the US Accidents dataset (Kaggle: "US Accidents")

| Syllabus unit | Section here |
|---|---|
| I Intro (batch size, learning rate, overfitting) | 4, 5 |
| II Architectures (FC network, CNN) | 4, 5, 6 |
| III Regularization (dropout, batch norm, ResNet skip connections) | 4, 6 |
| IV Pre-trained models (YOLO, ResNet fine-tuning) | 3, 4 |
| V GANs | future work only (written in the paper) |

## 0. Setup

In [ ]:
!pip -q install ultralytics
import os, random, json, glob, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay, roc_auc_score)
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

DATA_DIR = "C:/Users/DELL/traffic_project/traffic_project"
IMG_DIR = f"{DATA_DIR}/images"
OUT_DIR = f"{DATA_DIR}/results"
os.makedirs(OUT_DIR, exist_ok=True)
CLASSES = ["low", "medium", "high"]
RESULTS = {}   # everything we report in the paper is stored here


## 1. Load your hand labels and split the data (70 / 15 / 15)
The split is made **once** and reused everywhere, so no image is ever in both training and test.

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/labels.csv")
df["level"] = df["level"].str.strip().str.lower()
df = df[df["filename"].apply(lambda f: os.path.exists(f"{IMG_DIR}/{f}"))].reset_index(drop=True)
df["y"] = df["level"].map({c: i for i, c in enumerate(CLASSES)})
print(df["level"].value_counts())

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df["y"], random_state=SEED)
print("train/val/test:", len(train_df), len(val_df), len(test_df))
RESULTS["split"] = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}

## 2. Quick look at your data

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, c in zip(axs, CLASSES):
    row = df[df["level"] == c].iloc[0]
    ax.imshow(Image.open(f"{IMG_DIR}/{row['filename']}")); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_samples.png", dpi=150); plt.show()

## 3. Pre-trained YOLO: count vehicles, compute PCU, get a density level (Unit IV)
YOLOv8 is used **as it is** (pre-trained on COCO). PCU = passenger car units, a weight per vehicle type.
The two thresholds LOW and HIGH are chosen using the **training images only**, then checked on the **test images**.
Cite a source for your PCU weights in the paper (for example the Indian Roads Congress, IRC).

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolov8n.pt")
COCO = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
PCU = {"car": 1.0, "motorcycle": 0.5, "bus": 3.0, "truck": 3.0}

def detect(fname, save=False):
    r = yolo(f"{IMG_DIR}/{fname}", conf=0.25, verbose=False)[0]
    counts = {n: 0 for n in COCO.values()}
    for c in r.boxes.cls.tolist():
        if int(c) in COCO: counts[COCO[int(c)]] += 1
    if save:
        os.makedirs(f"{OUT_DIR}/annotated", exist_ok=True)
        r.save(filename=f"{OUT_DIR}/annotated/{fname}")
    return counts

rows = []
for i, f in enumerate(df["filename"]):
    c = detect(f, save=(i < 12))      # saves 12 annotated pictures for the report
    rows.append({"filename": f, **c, "pcu": sum(c[k] * PCU[k] for k in c)})
det = pd.DataFrame(rows).merge(df[["filename", "level", "y"]], on="filename")
det.to_csv(f"{OUT_DIR}/detections.csv", index=False)
det.head()

In [ ]:
def pcu_level(p, lo, hi): return 0 if p < lo else (1 if p < hi else 2)

tr = det[det["filename"].isin(train_df["filename"])]
best = (0, None)
for lo in range(1, 30):
    for hi in range(lo + 2, 80, 2):
        acc = np.mean([pcu_level(p, lo, hi) for p in tr["pcu"]] == tr["y"].values)
        if acc > best[0]: best = (acc, (lo, hi))
LOW, HIGH = best[1]
print("Thresholds chosen on train:", LOW, HIGH, "train acc:", round(best[0], 3))

te = det[det["filename"].isin(test_df["filename"])]
pcu_pred = np.array([pcu_level(p, LOW, HIGH) for p in te["pcu"]])
pcu_acc = accuracy_score(te["y"], pcu_pred)
pcu_f1 = f1_score(te["y"], pcu_pred, average="macro")
print(f"YOLO+PCU on test: accuracy {pcu_acc:.3f}, macro-F1 {pcu_f1:.3f}")
ConfusionMatrixDisplay(confusion_matrix(te["y"], pcu_pred, labels=[0, 1, 2]), display_labels=CLASSES).plot(cmap="Blues")
plt.title("YOLO + PCU vs hand labels (test)"); plt.savefig(f"{OUT_DIR}/fig_cm_yolo.png", dpi=150); plt.show()
RESULTS["yolo_pcu"] = {"low_thr": LOW, "high_thr": HIGH, "test_acc": pcu_acc, "test_macro_f1": pcu_f1}

## 4. Fine-tune a pre-trained ResNet18 to classify traffic level (Units II, III, IV)
ResNet uses **skip connections** (Unit III). Training in two stages (Unit IV):
1. only the new final layer is trained (backbone frozen),
2. the last block (`layer4`) is unfrozen and fine-tuned with a smaller learning rate.

The final layer has **dropout** (Unit III).

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class TrafficImages(Dataset):
    def __init__(self, frame, tf): self.f = frame["filename"].tolist(); self.y = frame["y"].tolist(); self.tf = tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        img = Image.open(f"{IMG_DIR}/{self.f[i]}").convert("RGB")
        return self.tf(img), self.y[i]

def loaders(bs):
    return (DataLoader(TrafficImages(train_df, tf_train), batch_size=bs, shuffle=True),
            DataLoader(TrafficImages(val_df, tf_eval), batch_size=32),
            DataLoader(TrafficImages(test_df, tf_eval), batch_size=32))

def make_resnet(dropout=0.3):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, 3))
    return m.to(device)

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    loss_fn = nn.CrossEntropyLoss()
    tot = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot / n, correct / n

def fit_resnet(lr, bs, ep_head=2, ep_ft=3, dropout=0.3):
    torch.manual_seed(SEED)
    trl, vl, _ = loaders(bs)
    m = make_resnet(dropout)
    h = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_state = -1, None
    def log(a, b):
        h["train_loss"].append(a[0]); h["train_acc"].append(a[1]); h["val_loss"].append(b[0]); h["val_acc"].append(b[1])
    # stage 1: new head only
    opt = torch.optim.Adam(m.fc.parameters(), lr=lr)
    for _ in range(ep_head):
        a = run_epoch(m, trl, opt); b = run_epoch(m, vl); log(a, b)
        if b[1] > best_acc: best_acc, best_state = b[1], {k: v.clone() for k, v in m.state_dict().items()}
    # stage 2: fine-tune layer4 too
    for p in m.layer4.parameters(): p.requires_grad = True
    opt = torch.optim.Adam([{"params": m.layer4.parameters(), "lr": lr / 10},
                            {"params": m.fc.parameters(), "lr": lr}])
    for _ in range(ep_ft):
        a = run_epoch(m, trl, opt); b = run_epoch(m, vl); log(a, b)
        if b[1] > best_acc: best_acc, best_state = b[1], {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    return m, h, best_acc
print("ResNet helpers ready")

## 5. Hyperparameter studies (Unit I)
Study 1: learning rate. Study 2: batch size (using the best learning rate). Selection uses the **validation** set only.

In [ ]:
runs = {}
for lr in [1e-2, 1e-3, 1e-4]:
    m, h, acc = fit_resnet(lr=lr, bs=16)
    runs[f"lr={lr}, bs=16"] = (m, h, acc, lr, 16)
    print(f"lr={lr}: best val acc {acc:.3f}")

best_lr = max([v for v in runs.values()], key=lambda v: v[2])[3]
for bs in [8, 32]:
    m, h, acc = fit_resnet(lr=best_lr, bs=bs)
    runs[f"lr={best_lr}, bs={bs}"] = (m, h, acc, best_lr, bs)
    print(f"bs={bs}: best val acc {acc:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, h, _, _, _) in runs.items():
    axs[0].plot(h["val_loss"], label=name); axs[1].plot(h["val_acc"], label=name)
axs[0].set_title("Validation loss"); axs[1].set_title("Validation accuracy")
for a in axs: a.set_xlabel("epoch"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_hparam.png", dpi=150); plt.show()

RESULTS["hparam_study"] = {k: {"best_val_acc": v[2]} for k, v in runs.items()}

### Final ResNet: pick the best run by validation accuracy, test it once

In [ ]:
best_name = max(runs, key=lambda k: runs[k][2])
model, hist, _, _, _ = runs[best_name]
print("Best configuration:", best_name)

# training vs validation curves (shows overfitting / underfitting)
plt.figure(figsize=(6, 4))
plt.plot(hist["train_loss"], label="train"); plt.plot(hist["val_loss"], label="validation")
plt.axvline(4.5, ls="--", c="gray"); plt.text(4.7, max(hist["train_loss"]) * 0.9, "unfreeze layer4", fontsize=8)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("ResNet18 fine-tuning")
plt.savefig(f"{OUT_DIR}/fig_resnet_curves.png", dpi=150); plt.show()

_, _, test_loader = loaders(16)
model.eval(); preds, trues, probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        p = torch.softmax(model(x.to(device)), 1).cpu()
        preds += p.argmax(1).tolist(); trues += y.tolist(); probs += p.tolist()
res_acc = accuracy_score(trues, preds); res_f1 = f1_score(trues, preds, average="macro")
print(f"ResNet18 test: accuracy {res_acc:.3f}, macro-F1 {res_f1:.3f}")
print(classification_report(trues, preds, target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(trues, preds, labels=[0, 1, 2]), display_labels=CLASSES).plot(cmap="Blues")
plt.title("ResNet18 (test)"); plt.savefig(f"{OUT_DIR}/fig_cm_resnet.png", dpi=150); plt.show()
RESULTS["resnet18"] = {"config": best_name, "test_acc": res_acc, "test_macro_f1": res_f1}

# examples the model got wrong (useful for the discussion section)
wrong = [i for i, (p, t) in enumerate(zip(preds, trues)) if p != t][:6]
if wrong:
    fig, axs = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3.2), squeeze=False)
    for ax, i in zip(axs[0], wrong):
        ax.imshow(Image.open(f"{IMG_DIR}/{test_df.iloc[i]['filename']}")); ax.axis("off")
        ax.set_title(f"true {CLASSES[trues[i]]}\npred {CLASSES[preds[i]]}", fontsize=9)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_errors.png", dpi=150); plt.show()

## 6. Accident-risk network on US Accidents (Units I, II, III)
A fully connected network. The US Accidents data only contains accidents, so the task is:
**given the time, weather and road features, predict whether the accident is severe (Severity 3 or 4).**
Describe it that way in the paper ("accident severity risk").

Regularization comparison (Unit III): no regularization, dropout only, batch norm only, both.

In [ ]:
acc_df = pd.read_csv(f"{DATA_DIR}/accidents.csv")
if len(acc_df) > 100000: acc_df = acc_df.sample(100000, random_state=SEED)
acc_df = acc_df.reset_index(drop=True)

t = pd.to_datetime(acc_df["Start_Time"], format="mixed", errors="coerce")
X = pd.DataFrame({"hour": t.dt.hour, "dayofweek": t.dt.dayofweek, "month": t.dt.month})
for c in ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]:
    if c in acc_df: X[c] = acc_df[c]
for c in ["Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", "Roundabout",
          "Station", "Stop", "Traffic_Calming", "Traffic_Signal"]:
    if c in acc_df: X[c] = acc_df[c].astype(int)
if "Sunrise_Sunset" in acc_df: X["is_night"] = (acc_df["Sunrise_Sunset"] == "Night").astype(int)
if "Weather_Condition" in acc_df:
    top = acc_df["Weather_Condition"].value_counts().index[:10]
    X = pd.concat([X, pd.get_dummies(acc_df["Weather_Condition"].where(acc_df["Weather_Condition"].isin(top), "Other"), prefix="wx").astype(int)], axis=1)
y = (acc_df["Severity"] >= 3).astype(int).values
print("features:", X.shape[1], "| severe share:", round(y.mean(), 3))

Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=SEED)
med = Xtr.median(); Xtr, Xva, Xte = Xtr.fillna(med), Xva.fillna(med), Xte.fillna(med)
mu, sd = Xtr.mean(), Xtr.std().replace(0, 1)          # input normalization (Unit III)
to_t = lambda a: torch.tensor(((a - mu) / sd).values, dtype=torch.float32)
Xtr_t, Xva_t, Xte_t = to_t(Xtr), to_t(Xva), to_t(Xte)
ytr_t, yva_t, yte_t = [torch.tensor(a, dtype=torch.float32) for a in (ytr, yva, yte)]
pos_weight = torch.tensor([(1 - ytr.mean()) / ytr.mean()], dtype=torch.float32).to(device)
print("train/val/test:", len(Xtr), len(Xva), len(Xte))

In [ ]:
def make_mlp(d, use_bn, p):
    layers, prev = [], d
    for w in [128, 64]:
        layers.append(nn.Linear(prev, w))
        if use_bn: layers.append(nn.BatchNorm1d(w))
        layers.append(nn.ReLU())
        if p > 0: layers.append(nn.Dropout(p))
        prev = w
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers).to(device)

def train_mlp(use_bn, p, epochs=30, lr=1e-3, bs=256):
    torch.manual_seed(SEED)
    m = make_mlp(Xtr_t.shape[1], use_bn, p)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=bs, shuffle=True)
    h = {"train_loss": [], "val_loss": []}
    best, best_state = 1e9, None
    for _ in range(epochs):
        m.train(); tot = 0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(m(xb).squeeze(1), yb)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(yb)
        m.eval()
        with torch.no_grad():
            vl = loss_fn(m(Xva_t.to(device)).squeeze(1), yva_t.to(device)).item()
        h["train_loss"].append(tot / len(Xtr_t)); h["val_loss"].append(vl)
        if vl < best: best, best_state = vl, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    return m, h

def test_mlp(m):
    m.eval()
    with torch.no_grad(): pr = torch.sigmoid(m(Xte_t.to(device)).squeeze(1)).cpu().numpy()
    return {"acc": accuracy_score(yte, pr > 0.5), "f1": f1_score(yte, pr > 0.5), "auc": roc_auc_score(yte, pr)}

configs = {"none": (False, 0.0), "dropout 0.3": (False, 0.3), "batchnorm": (True, 0.0), "dropout + batchnorm": (True, 0.3)}
mlp = {}
for name, (bn, p) in configs.items():
    m, h = train_mlp(bn, p); mlp[name] = (m, h, test_mlp(m)); print(name, mlp[name][2])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
h0 = mlp["none"][1]
axs[0].plot(h0["train_loss"], label="train"); axs[0].plot(h0["val_loss"], label="validation")
axs[0].set_title("No regularization: train vs validation loss"); axs[0].legend()
for name, (_, h, _) in mlp.items(): axs[1].plot(h["val_loss"], label=name)
axs[1].set_title("Validation loss with / without regularization"); axs[1].legend()
for a in axs: a.set_xlabel("epoch"); a.set_ylabel("loss")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_regularization.png", dpi=150); plt.show()

table = pd.DataFrame({k: v[2] for k, v in mlp.items()}).T.round(3)
print(table); table.to_csv(f"{OUT_DIR}/table_regularization.csv")
RESULTS["accident_mlp"] = {k: v[2] for k, v in mlp.items()}

## 7. Save everything for the paper and PPT

In [ ]:
final = pd.DataFrame([]).round(3)
print(final); final.to_csv(f"{OUT_DIR}/table_traffic_level.csv", index=False)
json.dump(RESULTS, open(f"{OUT_DIR}/results.json", "w"), indent=2, default=float)

zp = f"{OUT_DIR}/results.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in glob.glob(f"{OUT_DIR}/**/*", recursive=True):
        if os.path.isfile(f): z.write(f, os.path.relpath(f, OUT_DIR))